# Is your backtest real? Check it in 30 seconds

[TokIO](https://github.com/Kuzi-DEV/Tokio-ai) asks whether a backtest's profit is distinguishable from luck, once two things are allowed for:

1. **Serial dependence.** Positions held for weeks make daily P&L far from independent; a plain t-test counts every bar as fresh evidence.
2. **Your search.** The Sharpe on your screen is the best of every variant you tried. The best of twenty zero-edge strategies looks good by construction.

Run the cells top to bottom. No API key, no account. The outputs saved here are from a run on 2026-10-06; yours will move a little as new days come in.

In [1]:
%pip install -q "tokio-ai>=0.5" yfinance

## 1. A grid everyone has run: SMA crossovers on SPY

Nineteen fast/slow pairs, long when fast > slow and short otherwise, on 20 years of daily SPY. The position is decided at the close and held over the next bar, so there is no lookahead.

In [2]:
import pandas as pd
import yfinance as yf
import tokio_ai

close = yf.download("SPY", period="20y", auto_adjust=True, progress=False)["Close"].squeeze()
r = close.pct_change()

pnl, positions = {}, {}
for fast in (5, 10, 20, 50):
    for slow in (50, 100, 150, 200, 250):
        if fast >= slow:
            continue
        f, s = close.rolling(fast).mean(), close.rolling(slow).mean()
        pos = (f > s).astype(float) * 2 - 1          # +1 long, -1 short
        pos = pos.where(s.notna()).shift(1)           # known at the close, held over the next bar
        name = f"sma{fast}/{slow}"
        pnl[name], positions[name] = pos * r, pos
print(len(pnl), "variants")

19 variants


## 2. The best one, the way it usually gets reported

In [3]:
best = max(pnl, key=lambda k: pnl[k].mean() / pnl[k].std())
print(best)
print(tokio_ai.check_backtest(pnl[best], positions=positions[best], asset_returns=r))

sma50/250


SIGNIFICANT (p=0.0372, alpha=0.05).
Sharpe 0.36 annualized over 4780 bars (~19.0 years at 252/year). A plain t-test would say p=0.0562; allowing for serial dependence in the P&L, p=0.0372 before any correction for trials.
Probabilistic Sharpe Ratio: 0.97 allowing for serial dependence; 0.94 as published, which treats bars as independent (0.95 passes).
Minimum track record length at 95%: 3904 bars (have 4780).
It would pass as the best of at most 1 independent trial. If you tried more than that, it's not evidence.

No costs added here (returns are taken as already net of any fees they include); the strategy turns over 2.0x a year.
Breakeven: it stays significant up to 22.1 bps per unit traded. If your real costs, slippage included, are higher, it's not evidence.
Without its best and worst 48 bars, the Sharpe is 0.50 (from 0.36): still positive.
Made money in 6 of 8 equal time blocks.
Rebuilt from its positions (no costs) the Sharpe is 0.36; with every position taken one bar later, 0.35.

## 3. The same backtest, honestly: every variant you tried

Pass the whole grid. The verdict is about the best variant, corrected for all of them using how correlated they are. You also get the Deflated Sharpe Ratio, the probability of backtest overfitting (does the in-sample winner keep winning out of sample?) and robustness checks.

In [4]:
print(tokio_ai.check_backtest(pnl, positions=positions, asset_returns=r))

NOT SIGNIFICANT after correcting for 19 trials (Romano-Wolf, using their correlation) (p=0.1153, alpha=0.05). Strongest variant: sma10/100.
Sharpe 0.36 annualized over 4780 bars (~19.0 years at 252/year). A plain t-test would say p=0.0592; allowing for serial dependence in the P&L, p=0.0199 before any correction for trials.
Haircut Sharpe after 19 trials: 0.21.
Deflated Sharpe Ratio of sma50/250: 0.80 allowing for serial dependence; 0.76 as published, which treats bars as independent (0.95 passes).
It would pass as the best of at most 2 independent trials. If you tried more than that, it's not evidence.

variant     Sharpe   p naive   p alone     p adj  haircut  verdict
sma10/100     0.36    0.0592    0.0199    0.1153     0.21  not significant
sma5/100      0.32    0.0800    0.0321    0.1656     0.17  not significant
sma10/200     0.36    0.0609    0.0360    0.1792     0.18  not significant
sma50/250     0.36    0.0562    0.0372    0.1827     0.18  not significant
sma5/150      0.33   

## 4. With trading costs

`costs=0.0005` charges 5 bps per unit of position traded. The output also gives the **breakeven cost**: the most you could pay and still have a significant result.

In [5]:
print(tokio_ai.check_backtest(pnl, positions=positions, asset_returns=r, costs=0.0005))

NOT SIGNIFICANT after correcting for 19 trials (Romano-Wolf, using their correlation) (p=0.1419, alpha=0.05). Strongest variant: sma10/100.
Sharpe 0.34 annualized over 4780 bars (~19.0 years at 252/year). A plain t-test would say p=0.0676; allowing for serial dependence in the P&L, p=0.0256 before any correction for trials.
Haircut Sharpe after 19 trials: 0.19.
Deflated Sharpe Ratio of sma50/250: 0.77 allowing for serial dependence; 0.74 as published, which treats bars as independent (0.95 passes).
It would pass as the best of at most 1 independent trial. If you tried more than that, it's not evidence.

variant     Sharpe   p naive   p alone     p adj  haircut  verdict
sma10/100     0.34    0.0676    0.0256    0.1419     0.19  not significant
sma50/250     0.36    0.0586    0.0398    0.2008     0.17  not significant
sma10/200     0.35    0.0655    0.0407    0.2023     0.17  not significant
sma5/100      0.30    0.0951    0.0440    0.2135     0.14  not significant
sma50/200     0.34    

## 5. A backtest that cheats

This one goes long on days that close up, deciding at that day's close: it uses information it couldn't have had. Every significance test passes it. The one-bar delay check doesn't.

In [6]:
cheat = (r > 0).astype(float) * 2 - 1      # knows today's return when it takes today's position
print(tokio_ai.check_backtest((cheat * r).dropna(), positions=cheat[r.notna()], asset_returns=r.dropna()))

SIGNIFICANT (p=0.0000, alpha=0.05).
Sharpe 13.26 annualized over 5029 bars (~20.0 years at 252/year). A plain t-test would say p=0.0000; allowing for serial dependence in the P&L, p=0.0000 before any correction for trials.
Probabilistic Sharpe Ratio: 1.00 allowing for serial dependence; 1.00 as published, which treats bars as independent (0.95 passes).
Minimum track record length at 95%: 77 bars (have 5029).
It would pass as the best of over a million independent trials.

No costs added here (returns are taken as already net of any fees they include); the strategy turns over 254.0x a year.
Breakeven: it stays significant up to 73.4 bps per unit traded. If your real costs, slippage included, are higher, it's not evidence.
Without its best and worst 50 bars, the Sharpe is 16.14 (from 13.26): still positive.
Made money in 8 of 8 equal time blocks.
Rebuilt from its positions (no costs) the Sharpe is 13.26; with every position taken one bar later, -0.44. Most of the edge needs same-bar exec

## 6. Your own backtest

**From vectorbt or backtesting.py**, pass the object itself; returns and positions are read from it:

```python
pf = vbt.Portfolio.from_signals(close, entries, exits, fees=0.001)   # a whole grid works too
print(tokio_ai.check_backtest(pf))

stats = Backtest(data, MyStrategy, commission=0.001).run()
print(tokio_ai.check_backtest(stats))
```

**From TradingView**, download the Strategy Tester's "List of trades" and run:

```python
print(tokio_ai.check_backtest(tokio_ai.read_tradingview("List of trades.csv"), trials=30))
```

**From anything else**, upload a CSV with one column of per-bar returns per variant (a date column is skipped), then run the cell below. If you tried more variants than you're uploading, set `TRIALS` to the real number.

In [7]:
TRIALS = None   # e.g. 40 if you tried 40 things and kept these

try:
    from google.colab import files
    name = next(iter(files.upload()))
except ImportError:
    name = input("path to your CSV: ")
header = open(name, encoding="utf-8-sig").readline()
if "Trade #" in header:                      # a TradingView export
    print(tokio_ai.check_backtest(tokio_ai.read_tradingview(name), trials=TRIALS))
else:
    df = pd.read_csv(name).select_dtypes("number")
    print(tokio_ai.check_backtest(df if df.shape[1] > 1 else df.iloc[:, 0], trials=TRIALS))

Found something odd? [Open an issue](https://github.com/Kuzi-DEV/Tokio-ai/issues/new/choose); there's a template for statistical concerns. The false-positive rates behind every number here are measured and reproducible: [docs/calibration.md](https://github.com/Kuzi-DEV/Tokio-ai/blob/main/docs/calibration.md).